## Notebook34

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
it_city = pl.read_csv(ub + "data/it_cities.csv")
prov = funs.read_file(ub + "data/it_province.geojson")
covid = pl.read_csv(ub + "data/it_province_covid.csv")

### Questions

This is the last notebook built on the Italy family, so it brings back all
three tables at once. `it_city` is the 388-city point table (`city_name`,
`lon`, `lat`, `pop`); `prov` is the 107-province polygon table (`province`,
`geometry`); `covid` is the daily case series from notebook32, but today only
its `region` column matters. Every province in `covid` carries a `region`
name, one of Italy's 21 official statistical regions (21, not the usual 20,
because Trentino-Alto Adige's two autonomous provinces, Trento and Bolzano,
are tracked as their own regions). `covid` has no polygon of its own, so
region is a grouping that has to be joined onto `prov` before it can be
mapped. Print results unless a question says otherwise.

1. In the first block, build a point `geometry` for `it_city`, then project
both `it_city` and `prov` into EPSG:7791. In the second block, print
`it_city`'s `_crs` column; in the third, print `prov`'s, to confirm they agree.

In [ ]:
it_city = it_city.geo.points_from_xy("lon", "lat").geo.to_crs(7791)
prov = prov.geo.to_crs(7791)

In [ ]:
it_city.select(c._crs.first())

In [ ]:
prov.select(c._crs.first())

2. Buffer every city by 20 kilometers with `it_city.geo.buffer(20_000)` and
plot the result with `geom_map(alpha=0.3)` and `coord_fixed()`. Where do the
discs overlap the most, and where do they stay isolated?

In [ ]:
(
    ggplot(it_city.geo.buffer(20_000))
    .geom_map(alpha=0.3)
    .coord_fixed()
)

**Answer**:


3. Compare `it_city.geo.sjoin(prov)` to `it_city.geo.buffer(20_000).geo.sjoin(
prov)`. In the first block, print the row count of `it_city.geo.sjoin(prov)`;
in the second, print the row count of the buffered join. What does the jump
mean?

In [ ]:
it_city.geo.sjoin(prov).select(pl.len())

In [ ]:
it_city.geo.buffer(20_000).geo.sjoin(prov).select(pl.len())

**Answer**:


4. Group the buffered join from question 3 by `city_name` and print the five
cities with the most province matches.

In [ ]:
(
    it_city
    .geo.buffer(20_000)
    .geo.sjoin(prov)
    .group_by(c.city_name)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
    .head(5)
)

**Answer**:


5. In the first block, build a province-to-region lookup from `covid` (`select`
`region` and `province`, then `.unique()`) and check it is a clean key with
`.select(c.province.n_unique(), pl.len())`. In the second block, join it onto
`prov` and print how many provinces failed to match (`c.region.is_null().sum()`)
and how many distinct regions came through (`c.region.n_unique()`).

In [ ]:
region_map = covid.select(c.region, c.province).unique()
region_map.select(c.province.n_unique(), pl.len())

In [ ]:
prov = prov.join(region_map, on="province", how="left")
prov.select(
    n_missing = c.region.is_null().sum(),
    n_region = c.region.n_unique()
)

**Answer**:


6. Compute each province's area with `.geo.area`, then `.geo.dissolve(
by="region", aggfunc={"area": "sum"})` to merge the provinces into their 21
regions. Print the regions sorted by total area (in square kilometers,
dividing by 1e6), largest first.

In [ ]:
prov = prov.geo.area
region = prov.geo.dissolve(by="region", aggfunc={"area": "sum"})
(
    region
    .select(c.region, (c.area / 1e6).round(0).alias("area_km2"))
    .sort(c.area_km2, descending=True)
)

**Answer**:


7. In the first block, plot the dissolved `region` table with `geom_map(fill=
"white", color="black")` and `coord_fixed()`. In the second block, print the
row count of `region` to compare against `prov`'s. What happened to the
province borders?

In [ ]:
(
    ggplot(region)
    .geom_map(fill="white", color="black")
    .coord_fixed()
)

In [ ]:
region.select(pl.len())

**Answer**:


8. In the first block, print the row count of `prov`. In the second block,
explode the full `prov` table with `.geo.explode()` and print the new row
count. In the third block, group the exploded table by `province`, count
the parts, and print the five provinces with the most.

In [ ]:
prov.select(pl.len())

In [ ]:
prov.geo.explode().select(pl.len())

In [ ]:
(
    prov
    .geo.explode()
    .group_by(c.province)
    .agg(n = pl.len())
    .sort(c.n, descending=True)
    .head(5)
)

**Answer**:


9. Filter the exploded table to `province == "Napoli"`, recompute `.geo.area`,
and print the four largest parts in square kilometers. Do the sizes match
what is known about Naples's islands?

In [ ]:
(
    prov
    .filter(c.province == "Napoli")
    .geo.explode()
    .geo.area
    .select((c.area / 1e6).round(1).alias("area_km2"))
    .sort(c.area_km2, descending=True)
    .head(4)
)

**Answer**:


10. Filter the exploded table to `province == "Trieste"` instead, recompute
`.geo.area`, and print the five largest parts. Does a high part count always
mean real islands, the way it did for Naples?

In [ ]:
(
    prov
    .filter(c.province == "Trieste")
    .geo.explode()
    .geo.area
    .select((c.area / 1e6).round(2).alias("area_km2"))
    .sort(c.area_km2, descending=True)
    .head(5)
)

**Answer**:


11. In the first block, count total vertices in `prov` with
`.geo.count_coordinates()` (`.select(c.count_coordinates.sum())`). In the
second block, repeat the count after `.geo.simplify(5_000)`. In the third
block, plot the simplified provinces with `geom_map(fill="white", color=
"black")` and `coord_fixed()`.

In [ ]:
prov.geo.count_coordinates().select(c.count_coordinates.sum())

In [ ]:
prov.geo.simplify(5_000).geo.count_coordinates().select(c.count_coordinates.sum())

In [ ]:
(
    ggplot(prov.geo.simplify(5_000))
    .geom_map(fill="white", color="black")
    .coord_fixed()
)

**Answer**:


12. Check that every polygon in `prov` is actually valid with `.geo.is_valid`
before trusting anything computed above. Print how many are not.

In [ ]:
prov.geo.is_valid.select(n_invalid = (~c.is_valid).sum())

**Answer**:
